In [49]:
import os
import pandas as pd

folder = r"C:\Users\DELL\Documents\project_data\data"
file_path = os.path.join(folder, "processed_dataset.xls")

df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

# ✅ Correct split FIRST
students_df = df[df["participant"] == 1].copy()
teachers_df = df[df["participant"] == 2].copy()

# Remove unnecessary columns for teachers
teachers_df = teachers_df.drop(columns=[
    'MPSS1','MPSS2','MPSS3','MPSS4','MPSS5','MPSS6',
    'MPSS7','MPSS8','MPSS9','MPSS10','MPSS11','MPSS12',
    'MPSS_total','AVERAGE','RANK'
], errors='ignore')

# Remove unnecessary columns for students
students_df = students_df.drop(columns=[
    'OSLO1','OSLO2','OSLO3','OSLO_total',
    'service_year_teacher'
], errors='ignore')

# Fix null values in students
for col in students_df.select_dtypes(include='number'):
    students_df[col] = students_df[col].fillna(students_df[col].median())

for col in students_df.select_dtypes(include='object'):
    if not students_df[col].mode().empty:
        students_df[col] = students_df[col].fillna(students_df[col].mode()[0])
    else:
        students_df[col] = students_df[col].fillna("Unknown")

# Save files
students_df.to_csv(os.path.join(folder, "students_participant1.csv"), index=False)
teachers_df.to_csv(os.path.join(folder, "teachers_participant2.csv"), index=False)

print("Processing completed successfully.")

Processing completed successfully.


In [50]:
folder = r"C:\Users\DELL\Documents\project_data\data"
file_path = os.path.join(folder, "teachers_participant2.csv")

# IMPORTANT: specify engine for .xls
df1 = pd.read_csv(file_path)

# Clean column names
df1.columns = df1.columns.str.strip()

print("Columns:", df1.columns)
print("Shape:", df1.shape)

Columns: Index(['ID', 'participant', 'age', 'sex', 'Education', 'average', 'rank',
       'school', 'service_year_teacher', 'SRQ1', 'SRQ2', 'SRQ3', 'SRQ4',
       'SRQ5', 'SRQ6', 'SRQ7', 'SRQ8', 'SRQ9', 'SRQ10', 'SRQ11', 'SRQ12',
       'SRQ13', 'SRQ14', 'SRQ15', 'SRQ16', 'SRQ17', 'SRQ18', 'SRQ19', 'SRQ20',
       'Alcohol_1', 'tobaco_1', 'khat_1', 'OSLO1', 'OSLO2', 'OSLO3',
       'SRQ_total', 'OSLO_total'],
      dtype='object')
Shape: (138, 37)


In [51]:
folder = r"C:\Users\DELL\Documents\project_data\data"
file_path = os.path.join(folder, "students_participant1.csv")

# IMPORTANT: specify engine for .xls
df1 = pd.read_csv(file_path)

# Clean column names
df1.columns = df1.columns.str.strip()
print("Columns:", df1.columns)
print("Shape:", df1.shape)

Columns: Index(['ID', 'participant', 'age', 'sex', 'Education', 'average', 'rank',
       'school', 'SRQ1', 'SRQ2', 'SRQ3', 'SRQ4', 'SRQ5', 'SRQ6', 'SRQ7',
       'SRQ8', 'SRQ9', 'SRQ10', 'SRQ11', 'SRQ12', 'SRQ13', 'SRQ14', 'SRQ15',
       'SRQ16', 'SRQ17', 'SRQ18', 'SRQ19', 'SRQ20', 'Alcohol_1', 'tobaco_1',
       'khat_1', 'MPSS1', 'MPSS2', 'MPSS3', 'MPSS4', 'MPSS5', 'MPSS6', 'MPSS7',
       'MPSS8', 'MPSS9', 'MPSS10', 'MPSS11', 'MPSS12', 'SRQ_total',
       'MPSS_total'],
      dtype='object')
Shape: (620, 45)


In [52]:
# -----------------------------
# CREATE OVERVIEW DASHBOARD DATASET
# -----------------------------

print("\n🟦 Creating Overview Dashboard Dataset...")

# -----------------------------
# ADD ROLE COLUMN
# -----------------------------
students_df["ROLE"] = "Student"
teachers_df["ROLE"] = "Teacher"

# -----------------------------
# CREATE UNIFIED SOCIAL SUPPORT COLUMN
# Students -> MPSS_TOTAL
# Teachers -> OSLO_TOTAL
# -----------------------------
students_df["SOCIAL_SUPPORT_TOTAL"] = students_df["MPSS_total"]
teachers_df["SOCIAL_SUPPORT_TOTAL"] = teachers_df["OSLO_total"]

# -----------------------------
# CREATE AGE GROUP
# -----------------------------
def age_group(age):
    if age <= 18:
        return "Teen"
    elif age <= 25:
        return "Young Adult"
    elif age <= 40:
        return "Adult"
    else:
        return "Senior"

students_df["AGE_GROUP"] = students_df["age"].apply(age_group)
teachers_df["AGE_GROUP"] = teachers_df["age"].apply(age_group)

# -----------------------------
# CREATE SRQ RISK LEVEL
# -----------------------------
def srq_risk(score):
    if score <= 10:
        return "Low Risk"
    elif score <= 15:
        return "Medium Risk"
    else:
        return "High Risk"

students_df["SRQ_RISK_LEVEL"] = students_df["SRQ_total"].apply(srq_risk)
teachers_df["SRQ_RISK_LEVEL"] = teachers_df["SRQ_total"].apply(srq_risk)

# -----------------------------
# SELECT IMPORTANT COLUMNS ONLY
# (what Overview page needs)
# -----------------------------
overview_cols = [
    "ID",
    "ROLE",
    "sex",
    "age",
    "AGE_GROUP",
    "Education",
    "school",
    "SRQ_total",
    "SOCIAL_SUPPORT_TOTAL",
    "SRQ_RISK_LEVEL"
]

students_overview = students_df[overview_cols]
teachers_overview = teachers_df[overview_cols]

# -----------------------------
# COMBINE INTO ONE TABLE
# -----------------------------
overview_df = pd.concat([students_overview, teachers_overview], ignore_index=True)

# -----------------------------
# SAVE OVERVIEW FILE
# -----------------------------
overview_path = os.path.join(folder, "overview_dashboard.csv")
overview_df.to_csv(overview_path, index=False)

print("✅ overview_dashboard.csv created successfully")
print(f"📊 Overview rows: {len(overview_df)}")
print("🎉 Overview dataset ready for Power BI")


🟦 Creating Overview Dashboard Dataset...
✅ overview_dashboard.csv created successfully
📊 Overview rows: 758
🎉 Overview dataset ready for Power BI
